In [2]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-09-09 17:28:13--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-09-09 17:28:13 (37.5 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [3]:
with open('input.txt', 'r', encoding = 'utf-8') as f:
    text = f.read()


In [ ]:
len(text)

1115394

In [ ]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [ ]:
stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode('hola senor'))
print(decode(encode('hola senor')))

[46, 53, 50, 39, 1, 57, 43, 52, 53, 56]
hola senor


In [ ]:
import torch
data = torch.tensor(encode(text))
print(data.shape, data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [ ]:
n = int(len(data) * 0.9)
train_data = data[:n]
val_data = data[n:]

In [ ]:
len(val_data), len(train_data)

(111540, 1003854)

In [ ]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [ ]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]

    print(f"when input is {context}, the target is {target}")

when input is tensor([18]), the target is 47
when input is tensor([18, 47]), the target is 56
when input is tensor([18, 47, 56]), the target is 57
when input is tensor([18, 47, 56, 57]), the target is 58
when input is tensor([18, 47, 56, 57, 58]), the target is 1
when input is tensor([18, 47, 56, 57, 58,  1]), the target is 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]), the target is 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]), the target is 58


In [ ]:
torch.manual_seed(6767)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])  #torch.stack just turns it into a tensor in this case.
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch("train")
print(xb.shape)
print(xb)

print(yb.shape)
print(yb)

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]

        print(f"when za context is {context.tolist()}, the target is {target}")

torch.Size([4, 8])
tensor([[53, 56, 52,  8,  1, 20, 43, 56],
        [39, 41, 46,  1, 58, 46, 43, 43],
        [61, 43, 56,  1, 44, 56, 53, 51],
        [57, 53,  6,  1, 40, 43, 47, 52]])
torch.Size([4, 8])
tensor([[56, 52,  8,  1, 20, 43, 56, 43],
        [41, 46,  1, 58, 46, 43, 43,  1],
        [43, 56,  1, 44, 56, 53, 51,  1],
        [53,  6,  1, 40, 43, 47, 52, 45]])
when za context is [53], the target is 56
when za context is [53, 56], the target is 52
when za context is [53, 56, 52], the target is 8
when za context is [53, 56, 52, 8], the target is 1
when za context is [53, 56, 52, 8, 1], the target is 20
when za context is [53, 56, 52, 8, 1, 20], the target is 43
when za context is [53, 56, 52, 8, 1, 20, 43], the target is 56
when za context is [53, 56, 52, 8, 1, 20, 43, 56], the target is 43
when za context is [39], the target is 41
when za context is [39, 41], the target is 46
when za context is [39, 41, 46], the target is 1
when za context is [39, 41, 46, 1], the target is 

In [ ]:
from torch import nn
from torch.nn import functional as F
torch.manual_seed(6767)

class BigramModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding(idx)

        if targets is None:
            loss = None

        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim = -1)

            idx_next = torch.multinomial(probs, num_samples =1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx
model = BigramModel(vocab_size)

logits, loss = model(xb, yb)
print(logits.shape)
print(loss)

decode(model.generate(idx = torch.zeros((1,1), dtype = torch.long), max_new_tokens = 50)[0].tolist())

torch.Size([256, 65])
tensor(4.6587, grad_fn=<NllLossBackward0>)
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 65])
torch.Size([1, 

'\n QwIcOgN!QDkxP,BePTTN?tww DWzgH zpZr\naMAMvuhFr3,cW'

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr = 1e-3)

In [ ]:
batch_size = 32
for steps in range(10000):
    xb, yb= get_batch('train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad()

    loss.backward()
    optimizer.step()

print(loss.item())
print(logits.shape)


In [ ]:
print(vocab_size)

65


In [ ]:
print(decode(model.generate(idx = torch.zeros((1,1), dtype = torch.long), max_new_tokens=400)[0].tolist()))


V:

AMy b

sshie ive,
ER:
An ty u con fisu,


ARGBEng ore! neporiverrod tekn SI e haif bou athiseve yoverjjow Anoit hir.
Wtin t, Bo thicomp.
ANI,
METhit od tiswe hou by Cous,
Che y BEven, s oued,
Bxfr lessud d wes t,
Anowhais st sok! I opldillew ss, Bund haul wiches?
Fo:
Havembe linoly.'s matt me:
Titheeds yr f des:

'HOf p panur it ind w Mownodr th scard!
SS:
'Whe hofut andyout, hiserswat Wod cea


In [ ]:
B, T, C = 4,8,2
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [ ]:
xbow = torch.zeros((B,T,C)) # bow is bag of words

for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]
        xbow[b,t] = torch.mean(xprev, 0)



In [ ]:
# doing the same thing as the upper cell, just way more efficient
wei  = torch.tril(torch.ones(T,T))
wei = wei / torch.sum(wei, 1, keepdim=True)
xbow2 = wei @ x #  the b here is auto-summoned by pytorch in wei(B,T,T) @ (B, T, C) -> (B,T,C)
torch.allclose(xbow, xbow2)

True

In [ ]:
# another cool shitt
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros(T,T)
wei = wei.masked_fill(tril == 0, float('-inf')) #stuff i learned from 3blue1brown
wei = F.softmax(wei, dim=1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)

True

we do the above stuff, so that the word/char at a position t cant have access to information of the words/chars ahead of that particular word/char, only having access to the informatin precedding t

In [ ]:
torch.manual_seed(6768)
a = torch.tril(torch.ones((3,3)))
# tril is a crazy cool func if you really think about it -- pretty useful when you matmul using it
a = a / torch.sum(a, 1, keepdim=True) # using it to avg in an incremental way
b = torch.randint(0,10, (3,2)).float()
c = a @ b
print(a)
print(b)
print(c)

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
tensor([[5., 7.],
        [1., 1.],
        [2., 5.]])
tensor([[5.0000, 7.0000],
        [3.0000, 4.0000],
        [2.6667, 4.3333]])


In [ ]:
torch.manual_seed(6767)
B, T, C = 4, 8, 32
x = torch.randn(B,T,C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x) # (B,T,16)
q = query(x) # (B,T,16)

#now focuses on a token based on learned params instead of giving equal importance to each token
wei = q @ k.transpose(-2, -1)  # (B, T, 16) @ (B, 16 , T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
# wei = torch.zeros(T, T)
wei = wei.masked_fill(tril == 0, float('-inf')) # no peeking foward hehe
wei = F.softmax(wei, dim=1)
# out = wei @ x

v = value(x)
out = wei @ v # the v gets aggregated
out.shape

torch.Size([4, 8, 16])

In [7]:
#all the em-dashes are user generated.

import torch
from torch import nn
from torch.nn import functional as F

batch_size = 64
block_size = 256 #context length
max_iters = 5000
eval_interval = 500
lr = 3e-4
device = "cuda" if torch.cuda.is_available() else "cpu"
eval_iters = 200
n_embed = 384
n_head = 6
n_layers = 6
dropout = .2
# head_size = 16

# !wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

with open('input.txt', 'r', encoding = 'utf-8') as f:
    text = f.read()

torch.manual_seed(1337)


chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text))



n = int(len(data) * 0.9)
train_data = data[:n]
val_data = data[n:]


def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])  #torch.stack just turns it into a tensor in this case.
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

xb, yb = get_batch("train")


@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch(split)
            _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    def __init__(self,head_size):
        super().__init__()
        self.head_size = head_size
        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x) # (B,T,head_size)
        q = self.query(x) # (B,T,head_size)

        #now focuses on a token based on learned params instead of giving equal importance to each token
        wei = (q @ k.transpose(-2, -1)) * (self.head_size ** -0.5)  # (B, T, head_size) @ (B, head_size , T) ---> (B, T, T)


        # wei = torch.zeros(T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0 , float('-inf')) # no peeking foward hehe, :T implemeted cause at the time of generation the T might be diff when starting
        wei = F.softmax(wei, dim=-1)  # the -inf elems gets the prob of 0 when passed thorugh softmax
        wei = self.dropout(wei)

        v = self.value(x)
        out = wei @ v # the v gets aggregated
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embed) # num_heads * head_size = n_embed. so, its 32 -> 32
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim = -1) #combining all the features to form 32 features again.
        out = self.proj(out)
        return self.dropout(out)

class FeedForward(nn.Module):
    def __init__(self, n_embed):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.ReLU(),
            nn.Linear(4 * n_embed, n_embed), #basicaly projection for ffwd, coupled straight into the sequential.
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embed, n_head):
        super().__init__()
        self.sa_head = MultiHeadAttention(n_head, n_embed//n_head)
        self.ffwd = FeedForward(n_embed)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)

    def forward(self, x):
        # adding residuals
        x = x + self.sa_head(self.ln1(x)) # we fork off, compute the sa and come back to the original pathway.
        x = x + self.ffwd(self.ln2(x))
        # we feed the x into LayerNorm before sending it to sa or ffwd.
        return x

class BigramModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embed)
        self.positional_embedding = nn.Embedding(block_size, n_embed)
        # self.sa_head = Head(n_embed)
        # self.sa_head = MultiHeadAttention(4, n_embed // 4)
        self.blocks = nn.Sequential(
            *[Block(n_embed, n_head) for _ in range(n_layers)]
        )
        self.ln = nn.LayerNorm(n_embed)
        # self.ffwd = FeedForward(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)
        pos_emb = self.positional_embedding(torch.arange(T, device = device ))
        x = tok_emb + pos_emb
        # x = self.sa_head(x)
        # x = self.ffwd(x)
        x = self.blocks(x)
        x = self.ln(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None

        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C) #converting the shape casue the cross_entropy doenst take 3 dims, so we're combining all the elements from diff batches into a single vector dim.
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        #!!!HEAVY LEARNING FROM HEAVY MISTAKE
        # idx_cropped = idx[:, -block_size:] -----> lol i tried to bring it up here thinking i was doing some crazy optimization -- welp, i was sending the autoregressive mech. on fire-- basically if it was out of the loop, it wont be able to use the newly generated tokens appended to it inside the loop -- so for every iter it would have just taken 0 as input basically "\n" xD
        for _ in range(max_new_tokens):
            idx_cropped = idx[:, -block_size:] #we have to crop it to only have 8 char because we have positional encoding
            #it will only have the last 8 elems/chars

            logits, loss = self(idx_cropped)
            logits = logits[:, -1, :] #taking only the last token, cause only the last token is used to generate the next token

            probs = F.softmax(logits, dim = -1)
             # will do softmax for each batch item -> here there is no sense of batch because the batch_size will be 1 but there is max_new_tokens. so, it would generate (1,vocab_size) softmax prob dist for every iter(basically for every new token)

            idx_next = torch.multinomial(probs, num_samples =1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx
model = BigramModel()
model = model.to(device)


optimizer = torch.optim.AdamW(model.parameters(), lr = lr) # AdamW is just adam-chan with weight decay decoupled from regular updatation.


for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f'step {iter} | Train loss {losses['train']:.4f} | val loss {losses['val']:.4f}')
    xb, yb= get_batch('train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad()

    loss.backward()
    optimizer.step()

print(loss.item())
context = torch.zeros((1,1), dtype = torch.long, device = device)
print(decode(model.generate(idx = context, max_new_tokens=2500)[0].tolist()))


step 0 | Train loss 4.3358 | val loss 4.3323
step 500 | Train loss 1.9022 | val loss 2.0187
step 1000 | Train loss 1.5449 | val loss 1.7288
step 1500 | Train loss 1.4022 | val loss 1.6138
step 2000 | Train loss 1.3139 | val loss 1.5521
step 2500 | Train loss 1.2569 | val loss 1.5277
step 3000 | Train loss 1.2067 | val loss 1.4993
step 3500 | Train loss 1.1639 | val loss 1.4869
step 4000 | Train loss 1.1276 | val loss 1.4799
step 4500 | Train loss 1.0913 | val loss 1.4747
1.1363441944122314

But with prison: I will steel win thou,
And now must I rest it of that writ,
To prevent a forcementy of it.

FLORIZEL:
My highness that I must be
The morelant's word: if you have been sworns.

POLIXENES:
My lords have had wonced me to the blind youth
Which noney I of you gaved the tidwardy.

CLAUS:
God sign; you say young him; I'lard yourself speak,
'Tis grows not the crown I pung his speak;'t well speak
To speak intellihelds with this cince? I'll rad aside.
The night of his right more in goldness
C

In [9]:
torch.save(model.state_dict(), 'shakespeare.pt')

In [8]:
print(sum(p.numel() for p in model.parameters())/1e6, 'M Params')

10.788929 M Params
